# 02 — COLMAP

COLMAPで、複数画像からcamera poseとスパースな3D構造を求めます。

`images → local features → image correspondences → camera poses + 3D points → undistorted cameras`の順に実行します。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from gs_tutorial.colmap import (
    ColmapPaths,
    colmap_database_summary,
    colmap_stage_status,
    export_processed_text,
    export_sparse_text,
    extract_features,
    map_sparse,
    match_features,
    read_feature_keypoints,
    read_strongest_verified_pair,
    resolve_backend,
    select_sparse_model,
    sparse_model_summary,
    undistort_images,
)
from gs_tutorial.config import load_config
from gs_tutorial.dataset import load_colmap_text, scene_summary
from gs_tutorial.visualization import (
    feature_keypoints_figure,
    sparse_scene_figure,
    verified_matches_figure,
)

cwd = Path.cwd().resolve()
repo = cwd.parent if (cwd.parent / "pyproject.toml").is_file() else cwd
config_path = repo / "configs/video.yaml"  # Use images.yaml for multi-view image input
cfg = load_config(config_path)
paths = ColmapPaths.create(repo / cfg.project_dir / "colmap", repo / cfg.project_dir / "images")
backend = resolve_backend("auto")
print("backend:", backend)
print("workspace:", paths.root)
colmap_stage_status(paths)

## 1. 特徴点抽出

特徴点抽出手法の1つであるSIFTは、別のviewでも見つけやすい点を`keypoint`として検出し、周囲の見え方を`descriptor`で表します。

画像上でkeypointが被写体と画像全体に分布しているか確認します。無地やblur領域では特徴が少なくなります。

In [ ]:
if colmap_stage_status(paths)["features"]:
    print("Feature extraction already complete")
else:
    extract_features(
        paths,
        backend=backend,
        camera_model=cfg.colmap.camera_model,
        single_camera=cfg.colmap.single_camera,
        use_gpu=cfg.colmap.use_gpu,
    )
colmap_database_summary(paths.database)

In [ ]:
feature_name, keypoints = read_feature_keypoints(paths.database)
feature_keypoints_figure(paths.image_dir / feature_name, keypoints);

## 2. 特徴点マッチング

descriptorが似たkeypointを対応付け、RANSACでcamera geometryと矛盾する対応を除きます。残った対応が`verified match`です。

対応線が同じ物理点を結び、画像全体に分布しているか確認します。`verified_pairs`と`verified_matches`が少ない場合は、画像間の重なりが不足しています。

In [ ]:
if colmap_stage_status(paths)["matching"]:
    print("Feature matching already complete")
else:
    match_features(paths, backend=backend, matcher=cfg.colmap.matcher, use_gpu=cfg.colmap.use_gpu)
colmap_database_summary(paths.database)

In [ ]:
name_a, name_b, keypoints_a, keypoints_b, matches = read_strongest_verified_pair(paths.database)
verified_matches_figure(
    paths.image_dir / name_a, paths.image_dir / name_b, keypoints_a, keypoints_b, matches
);

## 3. スパース再構成 (SfM)

対応の強い2画像から復元を始め、画像と3D点を順に追加します。最後にbundle adjustmentでcamera poseと3D点を調整します。

- `registered_images`: 一貫した3D座標系へ配置できた画像数
- `sparse_points`: 複数viewから三角測量できた特徴点数
- `mean_reprojection_error_px`: 推定3D点を画像へ戻した位置と観測位置の平均差

登録画像が多く、再投影誤差が小さいことを確認します。

In [ ]:
if colmap_stage_status(paths)["mapping"]:
    print("Mapping already complete")
    model_dir = select_sparse_model(paths)
else:
    model_dir = map_sparse(paths, backend=backend)
sparse_model_summary(model_dir)

再構成された3D点群と、色を撮影順に変化させたcamera frustumを可視化しています。
frustumが被写体へ向いて連続して配置されているか確認します。向きや位置の飛びは誤matchの兆候です。

In [ ]:
if not colmap_stage_status(paths)["sparse_text"]:
    export_sparse_text(paths, backend=backend, model_dir=model_dir)
mapped_scene = load_colmap_text(paths.sparse_text_dir)
print(scene_summary(mapped_scene))
sparse_scene_figure(mapped_scene).show()

## 4. 画像の歪み補正

スマートフォン画像にはradial distortionがあります。COLMAPの推定値を使い、画像とcamera parameterを歪みのないPINHOLE modelへ変換します。

左右比較で、主に画像周辺の画角や直線が変化することを確認します。

In [ ]:
if colmap_stage_status(paths)["undistortion"]:
    print("Image undistortion already complete")
else:
    undistort_images(paths, backend=backend, model_dir=model_dir)
preview_name = mapped_scene.images[0].name
original_path = paths.image_dir / preview_name
undistorted_path = paths.processed_dir / "images" / preview_name
figure, axes = plt.subplots(1, 2, figsize=(16, 6))
for axis, path, title in zip(axes, [original_path, undistorted_path], ["Original", "Undistorted"]):
    axis.imshow(plt.imread(path))
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()

## 5. 学習用modelの出力

COLMAP modelを学習コードから読みやすいtext形式へ変換します。推定結果自体は変わりません。

- `cameras.txt`: 画像sizeとcamera内部parameter
- `images.txt`: camera poseと2D観測
- `points3D.txt`: 3D位置、色、観測track

出力したmodelを再読込し、cameraと3D点を正しく読み込めること、全stageが完了していることを確認します。

In [ ]:
if not colmap_stage_status(paths)["processed_text"]:
    export_processed_text(paths, backend=backend)
scene = load_colmap_text(paths.text_model_dir)
print(scene_summary(scene))
print(colmap_stage_status(paths))